Task 2: Calculate Scale and Zero Point

File: task_02_scale_zero_point.ipynb

Objective
Calculate the quantization parameters (scale and zero point) required to map a floating-point tensor to the INT8 range. In Task 1, these were given; now you compute them yourself.

In [1]:
import torch

In [ ]:
def calculate_scale_zero_point(tensor: torch.Tensor, q_min: int = -128, q_max: int = 127):
    t_min, t_max = tensor.min().item(), tensor.max().item()
    t_range, q_range = t_max - t_min, q_max - q_min
    
    # for small range difference
    if t_range < 1e-11:
        scale = 1.0
    else:
        scale = t_range / q_range
    zero_point = torch.round(torch.tensor(q_min - (t_min / scale)))
    zero_point = int(zero_point.clamp(min=q_min, max=q_max))
    
    return scale, zero_point

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

ip_tensors = {"t1": [-1.5, -0.8, 0.0, 0.9, 2.3],
     "t2": [0.1, 0.5, 1.2, 2.0, 3.5],
     "t3": [-3.0, -2.1, -1.4, -0.6, -0.1],
     "t4": [5.0, 5.0, 5.0],
     "t5": [1e-9, 2e-9, -1e-9]}

In [12]:
def calculate_print_output(tensor:torch.tensor):
    t_min, t_max = tensor.min().item(), tensor.max().item()
    scale, zero_point = calculate_scale_zero_point(tensor)
    quantized_tensor = torch.quantize_per_tensor(tensor, scale, zero_point, dtype=torch.qint8)
    dequantized_tensor = torch.dequantize(quantized_tensor)
    loss = torch.nn.L1Loss()
    mae = loss(tensor, dequantized_tensor)
    print(f'''Tensor Values: {tensor}
Tensor min: {t_min}
Tensor max: {t_max}
Scale: {scale}
Zero point: {zero_point}
Quantized Tensor: {quantized_tensor}
Dequantized Tensor: {dequantized_tensor}
MAE: {mae}''')

In [13]:
for key in ip_tensors.keys():
    print(f"---Tensor {key}---")
    calculate_print_output(torch.tensor(ip_tensors[key]))    

---Tensor t1---
Tensor Values: tensor([-1.5000, -0.8000,  0.0000,  0.9000,  2.3000])
Tensor min: -1.5
Tensor max: 2.299999952316284
Scale: 0.014901960597318761
Zero point: -27
Quantized Tensor: tensor([-1.5051, -0.8047,  0.0000,  0.8941,  2.2949], size=(5,),
       dtype=torch.qint8, quantization_scheme=torch.per_tensor_affine,
       scale=0.014901960597318761, zero_point=-27)
Dequantized Tensor: tensor([-1.5051, -0.8047,  0.0000,  0.8941,  2.2949])
MAE: 0.004156863782554865
---Tensor t2---
Tensor Values: tensor([0.1000, 0.5000, 1.2000, 2.0000, 3.5000])
Tensor min: 0.10000000149011612
Tensor max: 3.5
Scale: 0.013333333327489741
Zero point: -128
Quantized Tensor: tensor([0.1067, 0.5067, 1.2000, 2.0000, 3.4000], size=(5,), dtype=torch.qint8,
       quantization_scheme=torch.per_tensor_affine, scale=0.013333333327489741,
       zero_point=-128)
Dequantized Tensor: tensor([0.1067, 0.5067, 1.2000, 2.0000, 3.4000])
MAE: 0.022666646167635918
---Tensor t3---
Tensor Values: tensor([-3.0000, -2

### Inference on Edge cases: 
- Handled the constant tensor case (scale = 0) by setting the scale value to 1 for that particular input.
- All positive and all negative range tensors have zero points correctly at -128 and 127 respectively, but the MAE is slightly higher compared to the other cases.
- Handled the low-precision inputs and restored the tensor with zero MAE, with the scale matching the expectations for the small range.